In [1]:
# NOTEBOOK NAME
# pyFLEXTRKRsandbox.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd

# from pathlib import Path      # used to play with pathnames to save

# from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# mapping things
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

# for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION AND PyFLEXTRKR FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions')
from CustomFunctions1 import *
sys.path.insert(0, '/scratch/v46/sg3241/tmp/PyFLEXTRKR/')  # folder *containing* pyflextrkr/
import logging
logging.basicConfig(level=logging.INFO)
from pyflextrkr.idcells_reflectivity import idcells_reflectivity
from pyflextrkr.tracksingle_driver import tracksingle_driver
from pyflextrkr.gettracks import gettracknumbers
from pyflextrkr.trackstats_driver import trackstats_driver

# for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

import glob

import shutil # I think this is to rename the output file from PyFLEXTRKR
import gc # something to prevent memory leaks
import warnings

# DASK things
from dask.distributed import LocalCluster, Client, as_completed
from tqdm.notebook import tqdm

from pathlib import Path      # used to play with pathnames to save

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

import matplotlib.colors as mcolors
import matplotlib.cm as cm

import time

# from matplotlib.patches import Circle # for radar range ring circles on the map
# from matplotlib.lines import Line2D # for plotting stars on the map legend

In [ ]:
StoragePath  = f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/QC2/Radar22/2024/20240101/SingleTimes/cellidfile_20240101_003500.nc'

FXR = xr.open_dataset(StoragePath)

FXR

In [ ]:
# FUNCTION (CHAD)
# STEP ZERO (SINGLE TIMES)
# DASK PARALLELIZED W/ PROGRESS BAR

def RadarFeatureExtraction(
    MinOfDay, RadarIDno, RadarYear, RadarMonth, RadarDay,
    QualityControlOption, LonShift
):
    """Extract PyFLEXTRKR features for a single radar timestep."""
    warnings.filterwarnings('ignore')

    YYYY = str(RadarYear).zfill(4)
    MM   = str(RadarMonth).zfill(2)
    DD   = str(RadarDay).zfill(2)

    RadarFileDate      = YYYY + MM + DD
    RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

    try:
        houri = MinOfDay // 60
        mini  = MinOfDay % 60
        RadarFileTime      = str(houri).zfill(2) + str(mini).zfill(2) + '00'
        RadarFileTimePrint = RadarFileTime[0:2] + ':' + RadarFileTime[2:4] + ':' + RadarFileTime[4:6]

        # --- Input path ---
        RadarGridPath = (
            f'/scratch/v46/sg3241/tmp/CustomRadarGrids/'
            f'QC{QualityControlOption}/Radar{RadarIDno}/{YYYY}/{YYYY}{MM}{DD}/'
            f'{RadarIDno}_{RadarFileDate}_{RadarFileTime}_QC{QualityControlOption}.nc'
        )
        
        if not os.path.exists(RadarGridPath):
            return f'⚠ Missing  {RadarFileTimePrint}: {RadarGridPath}'

        # --- Output path ---
        output_path = (
            f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/'
            f'QC{QualityControlOption}/Radar{RadarIDno}/{YYYY}/{YYYY}{MM}{DD}/SingleTimes/'
        )
        os.makedirs(output_path, exist_ok=True)

        # --- Extract scheduled date/time from filename ---
        input_basename = os.path.basename(RadarGridPath)
        parts          = input_basename.replace('.nc', '').split('_')
        scheduled_date = parts[1]
        scheduled_time = parts[2]

        # --- Skip if output already exists ---
        scheduled_outfile = output_path + f'cellidfile_{scheduled_date}_{scheduled_time}.nc'
        if os.path.exists(scheduled_outfile):
            return f'⚠ Skipped  {RadarFileTimePrint}: already exists'

        # --- Config ---
        pyflextrkr_config = {
            'input_format'          : 'netcdf',
            'reflectivity_varname'  : 'corrected_reflectivity',
            'time_dimname'          : 'time',
            'z_dimname'             : 'z',
            'y_dimname'             : 'y',
            'x_dimname'             : 'x',
            'x_coordname'           : 'x',
            'y_coordname'           : 'y',
            'z_coordname'           : 'z',
            'lon_coordname'         : 'lon',
            'lat_coordname'         : 'lat',
            'radar_lon_varname'     : 'radar_longitude',
            'radar_lat_varname'     : 'radar_latitude',
            'dx'                    : 1000.0,
            'dy'                    : 1000.0,
            'is_3d'                 : True,
            'z_coord_type'          : 'height',
            'sfc_dz_min'            : 500.0,
            'sfc_dz_max'            : 3000.0,
            'echotop_gap'           : 2,
            'default_sfc_height'    : 0.0,
            'radar_sensitivity'     : 0.0,
            'absConvThres'          : 30.0,
            'minZdiff'              : 10.0,
            'truncZconvThres'       : 42.0,
            'mindBZuse'             : 10.0,
            'dBZforMaxConvRadius'   : 45.0,
            'conv_rad_increment'    : 0.2,
            'conv_rad_start'        : 1.0,
            'bkg_refl_increment'    : 5.0,
            'maxConvRadius'         : 1.0,
            'radii_expand'          : [1, 2, 3, 4, 5],
            'weakEchoThres'         : 15.0,
            'bkgrndRadius'          : 11.0,
            'min_corearea'          : 4.0,
            'min_cellarea'          : 4.0,
            'tracking_outpath'      : output_path,
            'cloudid_filebase'      : 'cellidfile_tmp_',
            'fillval'               : -9999,
            'return_diag'           : False,
            'convolve_method'       : 'fft',
            'dilate_method'         : 'orig',
            'expand_method'         : 'orig',
            'echotop_method'        : 'orig',
            'remove_smallcores'     : True,
            'remove_smallcells'     : False,
        }

        # --- Run identification ---
        outfile = idcells_reflectivity(RadarGridPath, pyflextrkr_config)
        shutil.move(outfile, scheduled_outfile)

        return f'✓ Completed {RadarFileTimePrint}'

    except Exception as e:
        return f'✗ Failed    {RadarFileTimePrint}: {str(e)}'

    finally:
        gc.collect()


In [ ]:
# EXECUTION (CHAD)
# STEP ZERO (SINGLE TIMES)
# DASK PARALLELIZED W/ PROGRESS BAR

import logging
logging.getLogger('distributed.worker.memory').setLevel(logging.ERROR)

# CONFIGURATION
RadarIDno            = '22'
RadarYear            = 2024
RadarMonth           = 3
QualityControlOption = 2

LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# --- Radar site name lookup ---
RadarSiteNames = {
    '22'  : 'Mackay',
    '106' : 'Townsville',
    '66'  : 'Mt Staplyton (Brisbane)',
    '50'  : 'Marburg',
    '19'  : 'Cairns',
    '8'   : 'Gympie',
    '24'  : 'Bowen',
    '23'  : 'Gladstone',
    '74'  : 'Greenvale',
    '98'  : 'Taroom',
    '108' : 'Towoomba',
    '78'  : 'Weipa',
    '72'  : 'Emerald',
}
RadarSiteName = RadarSiteNames.get(RadarIDno, f'Site {RadarIDno}')

# --- Longitude shift correction ---
LonShift = 146.5505 - (-19.4195) if RadarSiteName == 'Townsville' else 0.0

# --- Parse time range ---
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour,   EndMin   = int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay   = EndHour   * 60 + EndMin

# --- Build task list — loop over days AND timesteps ---
tasks_args = [
    (MinOfDay, RadarIDno, RadarYear, RadarMonth, RadarDay, QualityControlOption, LonShift)
    for RadarDay in range(9, 9 + 1)
    for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5)
]

print(f'Total tasks queued: {len(tasks_args)}')

# --- Execute in parallel with progress bar ---
cluster = LocalCluster(
    n_workers          = 4,
    threads_per_worker = 1,
    memory_limit       = '6GB',
)
client = Client(cluster)

futures = [client.submit(RadarFeatureExtraction, *args) for args in tasks_args]

completed = []
skipped   = []
missing   = []
failed    = []

for future in tqdm(as_completed(futures), total=len(futures), desc='Feature Extraction Progress'):
    result = future.result()
    if '✓' in result:
        completed.append(result)
    elif 'Skipped' in result:
        skipped.append(result)
    elif 'Missing' in result:
        missing.append(result)
        print(result)  # print missing so you can inspect paths
    elif '✗' in result:
        failed.append(result)
        print(result)

print(f'\n--- Summary ---')
print(f'✓  Completed : {len(completed)}')
print(f'⚠  Skipped   : {len(skipped)}')
print(f'⚠  Missing   : {len(missing)}')
print(f'✗  Failed    : {len(failed)}')


client.close()


In [ ]:
# FUNCTION (CHAD)
# STEP ONE
# DASK PARALLELIZED W/ PROGRESS BAR
# 1 WORKER ASSIGNED PER DAY

def RadarTrackingStep1_SingleDay(
    DayTimestamp,
    RadarIDno,
    QualityControlOption,
    TrackingVersionNumber,
):
    """
    Run tracksingle_driver for a single day's worth of cellidfiles.
    Each call is independent and writes track_*.nc files to its own
    per-day staging directory.

    Parameters
    ----------
    DayTimestamp         : pd.Timestamp  e.g. pd.Timestamp('2024-02-01')
    RadarIDno            : str
    QualityControlOption : int
    TrackingVersionNumber: int
    """
    
    warnings.filterwarnings('ignore')
    logging.getLogger('pyflextrkr').setLevel(logging.DEBUG)

    date_str = DayTimestamp.strftime('%Y%m%d')
    year_int = DayTimestamp.year
    QC       = str(QualityControlOption)
    VER      = str(TrackingVersionNumber)
    RID      = str(RadarIDno)

    # --- Per-day staging directory (cellidfile symlinks already exist here) ---
    # We reuse the existing per-day cellidfile directory directly
    daily_cellid_dir = (f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{VER}/QC{QC}/Radar{RID}/{year_int}/{date_str}/SingleTimes/')

    # --- Per-day track output directory ---
    # track_*.nc files go here — isolated per day so workers don't collide
    daily_track_dir = (f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{VER}/QC{QC}/Radar{RID}/{year_int}/{date_str}/tracks/')
    os.makedirs(daily_track_dir, exist_ok=True)

    # --- Find cellidfiles for this day ---
    cellid_files = sorted(glob.glob(os.path.join(daily_cellid_dir, 'cellidfile_*.nc')))

    if len(cellid_files) == 0:
        return f'⚠ No cellidfiles for {date_str}, skipping'

    def datetime_from_cellid(filepath):
        basename = os.path.basename(filepath)
        parts    = basename.replace('.nc', '').split('_')
        return pd.Timestamp(
            f"{parts[1][:4]}-{parts[1][4:6]}-{parts[1][6:8]} "
            f"{parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}"
        )

    epoch          = pd.Timestamp('1970-01-01')
    t_first        = datetime_from_cellid(cellid_files[0])
    t_last         = datetime_from_cellid(cellid_files[-1])
    start_basetime = (t_first - epoch).total_seconds()
    end_basetime   = (t_last  - epoch).total_seconds()
    startdate_str  = t_first.strftime('%Y%m%d.%H%M%S')
    enddate_str    = t_last.strftime('%Y%m%d.%H%M%S')

    total_minutes      = (t_last - t_first).total_seconds() / 60.0
    max_duration_steps = int(total_minutes / 5) + 1

    config = {
        'feature_type'                  : 'radar_cells',
        'tracking_outpath'              : daily_cellid_dir,   # <-- isolated per day
        'stats_outpath'                 : daily_track_dir,   # not used in step 1
        'cloudid_filebase'              : 'cellidfile_',
        'singletrack_filebase'          : 'track_',
        'tracknumbers_filebase'         : 'tracknumbers_',
        'trackstats_filebase'           : 'trackstats_',
        'trackstats_sparse_filebase'    : 'trackstats_sparse_',
        'pixeltracking_filebase'        : 'celltracks_',
        'startdate'                     : startdate_str,
        'enddate'                       : enddate_str,
        'start_basetime'                : start_basetime,
        'end_basetime'                  : end_basetime,
        'time_format'                   : 'yyyymodd.hhmmss',
        'datatimeresolution'            : 5.0 / 60.0,
        'dx'                            : 1000.0,
        'dy'                            : 1000.0,
        'pixel_radius'                  : 1.0,
        'area_method'                   : 'fixed',
        'feature_varname'               : 'feature_number',
        'nfeature_varname'              : 'nfeatures',
        'featuresize_varname'           : 'npix_feature',
        'ref_varname'                   : 'dbz_comp',
        'x_dimname'                     : 'lon',
        'y_dimname'                     : 'lat',
        'time_dimname'                  : 'time',
        'timegap'                       : 0.25,
        'othresh'                       : 0.3,
        'nmaxlinks'                     : 10,
        'maxnclouds'                    : 1000,
        'fillval'                       : -9999,
        'duration_range'                : [4, max_duration_steps],
        'duration_range_auto_update'    : True,
        'duration_range_round_base'     : 1,
        'remove_shorttracks'            : 1,
        'trackstats_dense_netcdf'       : 1,
        'match_pixel_dt_thresh'         : 60.0,
        'tracks_dimname'                : 'tracks',
        'times_dimname'                 : 'times',
        'run_parallel'                  : 0,
        'terrain_file'                  : None,
        'rangemask_varname'             : None,
    }

    # t0 = time.time()
    # result = tracksingle_driver(config)
    # elapsed = time.time() - t0
    # n_tracks = len(glob.glob(os.path.join(daily_track_dir, 'track_*.nc')))
    # return (
    #     f'✓ {date_str}: {n_tracks} track files written ({elapsed:.1f}s)\n'
    #     f'  tracksingle_driver returned: {result}\n'
    #     f'  config tracking_outpath: {config["tracking_outpath"]}\n'
    #     f'  config cloudid_outpath:  {config.get("cloudid_outpath", "NOT SET")}\n'
    #     f'  track files found: {glob.glob(os.path.join(daily_track_dir, "track_*.nc"))}'
    # )

    try:
        t0 = time.time()
        tracksingle_driver(config)
        elapsed = time.time() - t0
        n_tracks = len(glob.glob(os.path.join(daily_track_dir, 'track_*.nc')))
        # n_tracks = len(glob.glob(os.path.join(daily_cellid_dir, 'track_*.nc')))
        return f'✓ {date_str}: {n_tracks} track files written ({elapsed:.1f}s)'

    except Exception as e:
        return f'✗ {date_str}: {str(e)}'

    finally:
        gc.collect()

In [ ]:
# EXECUTION (CHAD)
# STEP ONE
# DASK PARALLELIZED W/ PROGRESS BAR
# 1 WORKER ASSIGNED PER DAY

# CONFIGURATION
PeriodStart           = '2024-02-01UTC00:00:00'
PeriodEnd             = '2024-02-29UTC23:59:59'
RadarIDno             = '41'
QualityControlOption  = 2
TrackingVersionNumber = 5

# Build one task per day
date_range = pd.date_range(
    start = pd.Timestamp(PeriodStart.replace('UTC', ' ')).normalize(),
    end   = pd.Timestamp(PeriodEnd.replace('UTC', ' ')).normalize(),
    freq  = 'D',
)

tasks_args = [
    (day, RadarIDno, QualityControlOption, TrackingVersionNumber)
    for day in date_range
]

print(f'Total days queued: {len(tasks_args)}')

cluster = LocalCluster(
    n_workers          = len(tasks_args),   # one worker per day 
    threads_per_worker = 1,
    memory_limit       = '6GB',
)
client = Client(cluster)

futures = [client.submit(RadarTrackingStep1_SingleDay, *args) for args in tasks_args]

for future in tqdm(as_completed(futures), total=len(futures), desc='Step 1 Progress'):
    print(future.result())

client.close()


In [2]:
# FUNCTION (CHAD)
# STEP ONE — MIDNIGHT BRIDGE
# DASK PARALLELISED W/ PROGRESS BAR
# 1 WORKER ASSIGNED PER DAY BOUNDARY

def RadarTrackingStep1_MidnightBridge(
    DayTimestamp,
    RadarIDno,
    QualityControlOption,
    TrackingVersionNumber,
):
    """
    Create the missing track_*.nc file that links the last cellidfile
    of DayTimestamp (Day N) to the first cellidfile of DayTimestamp+1
    (Day N+1).

    A temporary 2-file staging directory is created, tracksingle_driver
    is run on it, the resulting track_*.nc file is moved into the Day N+1
    SingleTimes directory, and the temporary directory is cleaned up.

    Parameters
    ----------
    DayTimestamp         : pd.Timestamp  The LEFT day (Day N), e.g. pd.Timestamp('2024-01-01')
    RadarIDno            : str
    QualityControlOption : int
    TrackingVersionNumber: int
    """
    warnings.filterwarnings('ignore')
    logging.getLogger('pyflextrkr').setLevel(logging.DEBUG)

    QC  = str(QualityControlOption)
    VER = str(TrackingVersionNumber)
    RID = str(RadarIDno)

    day_n      = DayTimestamp.normalize()
    day_n1     = day_n + pd.Timedelta(days=1)

    date_str_n  = day_n.strftime('%Y%m%d')
    date_str_n1 = day_n1.strftime('%Y%m%d')
    year_n      = day_n.year
    year_n1     = day_n1.year

    # ------------------------------------------------------------------ #
    # 1. Locate the last cellidfile of Day N and first of Day N+1
    # ------------------------------------------------------------------ #
    dir_n  = (
        f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{VER}/QC{QC}/Radar{RID}/'
        f'{year_n}/{date_str_n}/SingleTimes/'
    )
    dir_n1 = (
        f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{VER}/QC{QC}/Radar{RID}/'
        f'{year_n1}/{date_str_n1}/SingleTimes/'
    )

    files_n  = sorted(glob.glob(os.path.join(dir_n,  'cellidfile_*.nc')))
    files_n1 = sorted(glob.glob(os.path.join(dir_n1, 'cellidfile_*.nc')))

    if len(files_n) == 0:
        return f'⚠ No cellidfiles for Day N  ({date_str_n}),  skipping bridge'
    if len(files_n1) == 0:
        return f'⚠ No cellidfiles for Day N+1 ({date_str_n1}), skipping bridge'

    last_file_n   = files_n[-1]    # last timestep of Day N
    first_file_n1 = files_n1[0]   # first timestep of Day N+1

    # ------------------------------------------------------------------ #
    # 2. Parse timestamps from filenames
    # ------------------------------------------------------------------ #
    def datetime_from_cellid(filepath):
        basename = os.path.basename(filepath)
        parts    = basename.replace('.nc', '').split('_')
        return pd.Timestamp(
            f"{parts[1][:4]}-{parts[1][4:6]}-{parts[1][6:8]} "
            f"{parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}"
        )

    t_last_n   = datetime_from_cellid(last_file_n)
    t_first_n1 = datetime_from_cellid(first_file_n1)

    # ------------------------------------------------------------------ #
    # 3. Determine the expected output filename BEFORE running anything
    #    (named after the later/second timestep, i.e. first file of Day N+1)
    # ------------------------------------------------------------------ #
    expected_track_name = (
        f'track_{date_str_n1}_{t_first_n1.strftime("%H%M%S")}.nc'
    )
    final_track_path = os.path.join(dir_n1, expected_track_name)

    if os.path.exists(final_track_path):
        return (
            f'⚠ Skipped  bridge {date_str_n} → {date_str_n1}: '
            f'{expected_track_name} already exists'
        )

    # ------------------------------------------------------------------ #
    # 4. Create temporary 2-file staging directory
    # ------------------------------------------------------------------ #
    bridge_dir = (
        f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{VER}/QC{QC}/Radar{RID}/'
        f'{year_n1}/{date_str_n1}/MidnightBridge_{date_str_n}_{date_str_n1}/'
    )
    os.makedirs(bridge_dir, exist_ok=True)

    # Symlink both files into the staging directory
    for src in [last_file_n, first_file_n1]:
        dst = os.path.join(bridge_dir, os.path.basename(src))
        if os.path.islink(dst):
            os.remove(dst)
        os.symlink(src, dst)

    # ------------------------------------------------------------------ #
    # 5. Build config
    # ------------------------------------------------------------------ #
    epoch          = pd.Timestamp('1970-01-01')
    start_basetime = (t_last_n   - epoch).total_seconds()
    end_basetime   = (t_first_n1 - epoch).total_seconds()
    startdate_str  = t_last_n.strftime('%Y%m%d.%H%M%S')
    enddate_str    = t_first_n1.strftime('%Y%m%d.%H%M%S')

    total_minutes      = (t_first_n1 - t_last_n).total_seconds() / 60.0
    max_duration_steps = max(2, int(total_minutes / 5) + 1)

    config = {
        'feature_type'                  : 'radar_cells',
        'tracking_outpath'              : bridge_dir,
        'stats_outpath'                 : bridge_dir,
        'cloudid_filebase'              : 'cellidfile_',
        'singletrack_filebase'          : 'track_',
        'tracknumbers_filebase'         : 'tracknumbers_',
        'trackstats_filebase'           : 'trackstats_',
        'trackstats_sparse_filebase'    : 'trackstats_sparse_',
        'pixeltracking_filebase'        : 'celltracks_',
        'startdate'                     : startdate_str,
        'enddate'                       : enddate_str,
        'start_basetime'                : start_basetime,
        'end_basetime'                  : end_basetime,
        'time_format'                   : 'yyyymodd.hhmmss',
        'datatimeresolution'            : 5.0 / 60.0,
        'dx'                            : 1000.0,
        'dy'                            : 1000.0,
        'pixel_radius'                  : 1.0,
        'area_method'                   : 'fixed',
        'feature_varname'               : 'feature_number',
        'nfeature_varname'              : 'nfeatures',
        'featuresize_varname'           : 'npix_feature',
        'ref_varname'                   : 'dbz_comp',
        'x_dimname'                     : 'lon',
        'y_dimname'                     : 'lat',
        'time_dimname'                  : 'time',
        'timegap'                       : 0.25,
        'othresh'                       : 0.3,
        'nmaxlinks'                     : 10,
        'maxnclouds'                    : 1000,
        'fillval'                       : -9999,
        'duration_range'                : [2, max_duration_steps],
        'duration_range_auto_update'    : True,
        'duration_range_round_base'     : 1,
        'remove_shorttracks'            : 1,
        'trackstats_dense_netcdf'       : 1,
        'match_pixel_dt_thresh'         : 60.0,
        'tracks_dimname'                : 'tracks',
        'times_dimname'                 : 'times',
        'run_parallel'                  : 0,
        'terrain_file'                  : None,
        'rangemask_varname'             : None,
    }

    # ------------------------------------------------------------------ #
    # 6. Run tracksingle_driver on the 2-file staging directory
    # ------------------------------------------------------------------ #
    try:
        t0 = time.time()
        tracksingle_driver(config)
        elapsed = time.time() - t0

        # ------------------------------------------------------------------ #
        # 7. Find the track file written into bridge_dir and move it to
        #    the Day N+1 SingleTimes directory
        # ------------------------------------------------------------------ #
        bridge_track_files = glob.glob(os.path.join(bridge_dir, 'track_*.nc'))

        if len(bridge_track_files) == 0:
            return (
                f'⚠ Bridge ran but no track_*.nc produced '
                f'({date_str_n} → {date_str_n1}, {elapsed:.1f}s) — '
                f'timegap may have been exceeded'
            )

        if len(bridge_track_files) > 1:
            # Unexpected — more than one track file from 2 input files
            # Take the one matching the expected name if possible,
            # otherwise take the last one alphabetically
            matched = [
                f for f in bridge_track_files
                if os.path.basename(f) == expected_track_name
            ]
            track_to_move = matched[0] if matched else sorted(bridge_track_files)[-1]
        else:
            track_to_move = bridge_track_files[0]

        shutil.move(track_to_move, final_track_path)

        return (
            f'✓ Bridge {date_str_n} → {date_str_n1}: '
            f'{os.path.basename(final_track_path)} written ({elapsed:.1f}s)'
        )

    except Exception as e:
        return f'✗ Bridge {date_str_n} → {date_str_n1}: {str(e)}'

    finally:
        # ------------------------------------------------------------------ #
        # 8. Always clean up the temporary staging directory
        # ------------------------------------------------------------------ #
        shutil.rmtree(bridge_dir, ignore_errors=True)
        gc.collect()


In [3]:
# EXECUTION (CHAD)
# STEP ONE — MIDNIGHT BRIDGE
# DASK PARALLELISED W/ PROGRESS BAR
# 1 WORKER ASSIGNED PER DAY BOUNDARY

# CONFIGURATION
PeriodStart           = '2024-01-01UTC00:00:00'
PeriodEnd             = '2024-01-31UTC23:59:59'
RadarIDno             = '19'
QualityControlOption  = 2
TrackingVersionNumber = 5

# --- Build one task per day BOUNDARY (Day N is the LEFT/earlier day) ---
# For a 31-day period Jan 01 → Jan 31, this produces 30 bridges:
#   Jan 01 → Jan 02, Jan 02 → Jan 03, ..., Jan 30 → Jan 31

t_start = pd.Timestamp(PeriodStart.replace('UTC', ' ')).normalize()
t_end   = pd.Timestamp(PeriodEnd.replace('UTC', ' ')).normalize()

# All days except the last (each day here is the LEFT side of a bridge)
bridge_days = pd.date_range(start=t_start, end=t_end - pd.Timedelta(days=1), freq='D')

tasks_args = [
    (day, RadarIDno, QualityControlOption, TrackingVersionNumber)
    for day in bridge_days
]

print(f'Total midnight bridges queued: {len(tasks_args)}')

# --- Execute in parallel ---
cluster = LocalCluster(
    n_workers          = len(tasks_args),
    threads_per_worker = 1,
    memory_limit       = '6GB',
)
client = Client(cluster)

futures = [client.submit(RadarTrackingStep1_MidnightBridge, *args) for args in tasks_args]

completed = []
skipped   = []
failed    = []

for future in tqdm(as_completed(futures), total=len(futures), desc='Midnight Bridge Progress'):
    result = future.result()
    print(result)
    if '✓' in result:
        completed.append(result)
    elif '⚠' in result:
        skipped.append(result)
    elif '✗' in result:
        failed.append(result)

print(f'\n--- Summary ---')
print(f'✓  Completed : {len(completed)}')
print(f'⚠  Skipped   : {len(skipped)}')
print(f'✗  Failed    : {len(failed)}')

client.close()


Total midnight bridges queued: 30


INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:43489
INFO:distributed.scheduler:  dashboard at:         /proxy/8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:38479'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:44631'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:40021'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:34469'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:46051'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:35471'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45555'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:38397'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:44239'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:44589'
INFO:distributed.nanny:        Start Nanny at: 'tcp

Midnight Bridge Progress:   0%|          | 0/30 [00:00<?, ?it/s]

INFO:distributed.core:Event loop was unresponsive in Nanny for 3.15s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Nanny for 3.15s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Nanny for 3.15s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Nanny for 3.15s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts and instability.
INFO:distributed.core:Event loop was unresponsive in Nanny for 3.15s.  This is often caused by long-running GIL-holding functions or moving large chunks of data. This can cause timeouts an

✓ Bridge 20240109 → 20240110: track_20240110_000000.nc written (295.9s)


INFO:pyflextrkr.tracksingle_drift:/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/QC2/Radar19/2024/20240104/MidnightBridge_20240103_20240104/track_20240104_000000.nc
INFO:pyflextrkr.tracksingle_driver:Done with tracking sequential pairs of idfeature files
INFO:pyflextrkr.tracksingle_drift:/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/QC2/Radar19/2024/20240116/MidnightBridge_20240115_20240116/track_20240116_000000.nc
INFO:pyflextrkr.tracksingle_driver:Done with tracking sequential pairs of idfeature files
INFO:pyflextrkr.tracksingle_drift:/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/QC2/Radar19/2024/20240111/MidnightBridge_20240110_20240111/track_20240111_000000.nc
INFO:pyflextrkr.tracksingle_driver:Done with tracking sequential pairs of idfeature files
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:34409 name: 27
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:34409
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:51256
I

... (17 outputs left out)

INFO:pyflextrkr.tracksingle_drift:/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/QC2/Radar19/2024/20240103/MidnightBridge_20240102_20240103/track_20240103_000000.nc
INFO:pyflextrkr.tracksingle_driver:Done with tracking sequential pairs of idfeature files


✓ Bridge 20240102 → 20240103: track_20240103_000000.nc written (106.4s)


INFO:pyflextrkr.tracksingle_drift:/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/QC2/Radar19/2024/20240120/MidnightBridge_20240119_20240120/track_20240120_000000.nc
INFO:pyflextrkr.tracksingle_driver:Done with tracking sequential pairs of idfeature files
INFO:distributed.scheduler:Remove client Client-b7328e01-e3cf-11f1-ac25-000000dafe80
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:52236; closing.
INFO:distributed.scheduler:Remove client Client-b7328e01-e3cf-11f1-ac25-000000dafe80
INFO:distributed.scheduler:Close client connection: Client-b7328e01-e3cf-11f1-ac25-000000dafe80


✓ Bridge 20240119 → 20240120: track_20240120_000000.nc written (34.0s)

--- Summary ---
✓  Completed : 30
⚠  Skipped   : 0
✗  Failed    : 0


In [4]:
# FUNCTIONS (CHAD)
# STEPS ONE, TWO, AND THREE 
# DASK PARALLELIZED W/ PROGRESS BAR
# 3 SEPARATED TRACKING FUNCTIONS

# ------------------------------------------------------------------ #
# STEP 1: Symlink staging + pairwise linking
# ------------------------------------------------------------------ #
def RadarTrackingStep1(
    PeriodStart,
    PeriodEnd,
    RadarIDno,
    QualityControlOption,
    TrackingVersionNumber,
):
    """
    Stage cellidfiles via symlinks and run pairwise linking
    (tracksingle_driver) across the full period.

    Returns the config dict so Steps 2 and 3 can reuse it without
    having to reconstruct it.
    """
    warnings.filterwarnings('ignore')
    logging.getLogger('pyflextrkr').setLevel(logging.INFO)

    # ------------------------------------------------------------------ #
    # 1. Parse start / end times
    # ------------------------------------------------------------------ #
    def parse_period_timestamp(ts_str):
        return pd.Timestamp(ts_str.replace('UTC', ' '), tz=None)

    t_start = parse_period_timestamp(PeriodStart)
    t_end   = parse_period_timestamp(PeriodEnd)

    if t_end <= t_start:
        raise ValueError(
            f'PeriodEnd ({PeriodEnd}) must be after PeriodStart ({PeriodStart}).'
        )

    start_label = t_start.strftime('%Y%m%d')
    end_label   = t_end.strftime('%Y%m%d')

    QC  = str(QualityControlOption)
    VER = str(TrackingVersionNumber)
    RID = str(RadarIDno)

    # ------------------------------------------------------------------ #
    # 2. Build paths
    # ------------------------------------------------------------------ #
    base_path = f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{VER}/QC{QC}/Radar{RID}/'

    staging_path = (
        f'{base_path}'
        f'Symlink_{start_label}_{end_label}/SingleTimes/'
    )
    stats_path = (
        f'{base_path}'
        f'Period_{start_label}_{end_label}/Stats/'
    )

    os.makedirs(staging_path, exist_ok=True)
    os.makedirs(stats_path,   exist_ok=True)

    # ------------------------------------------------------------------ #
    # 3. Discover and symlink cellidfiles across every day in the period
    # ------------------------------------------------------------------ #
    date_range = pd.date_range(
        start = t_start.normalize(),
        end   = t_end.normalize(),
        freq  = 'D',
    )

    def datetime_from_cellid(filepath):
        basename = os.path.basename(filepath)
        parts    = basename.replace('.nc', '').split('_')
        return pd.Timestamp(
            f"{parts[1][:4]}-{parts[1][4:6]}-{parts[1][6:8]} "
            f"{parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}"
        )

    all_symlinked = []
    missing_days  = []

    for day in date_range:
        date_str  = day.strftime('%Y%m%d')
        year_int = day.year
        daily_dir = f'{base_path}{year_int}/{date_str}/SingleTimes/'

        daily_files = sorted(
            glob.glob(os.path.join(daily_dir, 'cellidfile_*.nc'))
        )

        if len(daily_files) == 0:
            missing_days.append(date_str)
            print(f'  ⚠ File missing for {date_str} — no cellidfiles found in {daily_dir}')
            continue

        for src in daily_files:
            try:
                file_time = datetime_from_cellid(src)
            except (IndexError, ValueError):
                print(f'  ⚠ Could not parse timestamp from {os.path.basename(src)}, skipping.')
                continue

            if not (t_start <= file_time <= t_end):
                continue

            dst = os.path.join(staging_path, os.path.basename(src))
            if os.path.islink(dst):
                os.remove(dst)
            os.symlink(src, dst)
            all_symlinked.append(dst)

    if len(all_symlinked) == 0:
        raise RuntimeError(
            f'No cellidfiles found for the period {PeriodStart} → {PeriodEnd}. '
            f'Check that feature extraction has been run for this period.'
            f'{daily_dir}'
        )

    all_symlinked = sorted(all_symlinked)

    print(f'\n  Period  : {PeriodStart} → {PeriodEnd}')
    print(f'  Files   : {len(all_symlinked)} cellidfiles staged')
    if missing_days:
        print(f'  Missing : {len(missing_days)} day(s) had no cellidfiles')
    print()

    # ------------------------------------------------------------------ #
    # 4. Determine actual start / end times from staged files
    # ------------------------------------------------------------------ #
    epoch          = pd.Timestamp('1970-01-01')
    t_first        = datetime_from_cellid(all_symlinked[0])
    t_last         = datetime_from_cellid(all_symlinked[-1])
    start_basetime = (t_first - epoch).total_seconds()
    end_basetime   = (t_last  - epoch).total_seconds()
    startdate_str  = t_first.strftime('%Y%m%d.%H%M%S')
    enddate_str    = t_last.strftime('%Y%m%d.%H%M%S')

    total_minutes      = (t_last - t_first).total_seconds() / 60.0
    max_duration_steps = int(total_minutes / 5) + 1

    # ------------------------------------------------------------------ #
    # 5. Build and return config
    # ------------------------------------------------------------------ #
    config = {
        'feature_type'                  : 'radar_cells',
        'tracking_outpath'              : staging_path,
        'stats_outpath'                 : stats_path,
        'cloudid_filebase'              : 'cellidfile_',
        'singletrack_filebase'          : 'track_',
        'tracknumbers_filebase'         : 'tracknumbers_',
        'trackstats_filebase'           : 'trackstats_',
        'trackstats_sparse_filebase'    : 'trackstats_sparse_',
        'pixeltracking_filebase'        : 'celltracks_',
        'startdate'                     : startdate_str,
        'enddate'                       : enddate_str,
        'start_basetime'                : start_basetime,
        'end_basetime'                  : end_basetime,
        'time_format'                   : 'yyyymodd.hhmmss',
        'datatimeresolution'            : 5.0 / 60.0,
        'dx'                            : 1000.0,
        'dy'                            : 1000.0,
        'pixel_radius'                  : 1.0,
        'area_method'                   : 'fixed',
        'feature_varname'               : 'feature_number',
        'nfeature_varname'              : 'nfeatures',
        'featuresize_varname'           : 'npix_feature',
        'ref_varname'                   : 'dbz_comp',
        'x_dimname'                     : 'lon',
        'y_dimname'                     : 'lat',
        'time_dimname'                  : 'time',
        'timegap'                       : 0.25,
        'othresh'                       : 0.3,
        'nmaxlinks'                     : 10,
        'maxnclouds'                    : 220,
        'fillval'                       : -9999,
        'duration_range'                : [4, max_duration_steps],
        'duration_range_auto_update'    : True,
        'duration_range_round_base'     : 1,
        'remove_shorttracks'            : 1,
        'trackstats_dense_netcdf'       : 0,
        'match_pixel_dt_thresh'         : 60.0,
        'tracks_dimname'                : 'tracks',
        'times_dimname'                 : 'times',
        'run_parallel'                  : 0,
        'terrain_file'                  : None,
        'rangemask_varname'             : None,
    }

    # ------------------------------------------------------------------ #
    # 6. Run Step 1
    # ------------------------------------------------------------------ #
    try:
        t0 = time.time()
        print('  Step 1/3: Pairwise linking ...')
        tracksingle_driver(config)
        print(f'            Done ({time.time() - t0:.1f}s)')
    except Exception as e:
        print(f'\n  ✗ Step 1 failed: {str(e)}\n')
        gc.collect()
        raise   # re-raise so the caller knows Step 1 did not complete

    gc.collect()
    return config


# ------------------------------------------------------------------ #
# STEP 2: Assemble full tracks
# ------------------------------------------------------------------ #
def RadarTrackingStep2(config):
    """
    Assemble full tracks from pairwise link files (gettracknumbers).

    Parameters
    ----------
    config : dict
        The config dict returned by RadarTrackingStep1, or reconstructed
        manually using RadarTrackingBuildConfig.
    """
    warnings.filterwarnings('ignore')
    logging.getLogger('pyflextrkr').setLevel(logging.INFO)

    try:
        t0 = time.time()
        print('  Step 2/3: Assembling full tracks ...')
        gettracknumbers(config)
        print(f'            Done ({time.time() - t0:.1f}s)')
    except Exception as e:
        print(f'\n  ✗ Step 2 failed: {str(e)}\n')
        gc.collect()
        raise

    gc.collect()


# ------------------------------------------------------------------ #
# STEP 3: Track statistics + optional cleanup
# ------------------------------------------------------------------ #
def RadarTrackingStep3(config, cleanup_symlinks=None):
    """
    Compute track statistics (trackstats_driver) and optionally clean
    up the symlink staging directory.

    Parameters
    ----------
    config           : dict   Config dict from Step 1 or BuildConfig.
    cleanup_symlinks : bool or None
        True  — delete staging directory after run
        False — leave it in place
        None  — prompt the user interactively
    """
    warnings.filterwarnings('ignore')
    logging.getLogger('pyflextrkr').setLevel(logging.INFO)

    trackstats_outfile = None

    try:
        t0 = time.time()
        print('  Step 3/3: Computing track statistics ...')
        trackstats_outfile = trackstats_driver(config)
        print(f'            Done ({time.time() - t0:.1f}s)')
        print(f'\n  ✓ Tracking complete: {trackstats_outfile}\n')
    except Exception as e:
        print(f'\n  ✗ Step 3 failed: {str(e)}\n')
        gc.collect()
        raise

    finally:
        gc.collect()

    # Symlink cleanup
    staging_path = config['tracking_outpath']

    if cleanup_symlinks is None:
        response = input(
            f'  Delete staging symlink directory?\n'
            f'  {staging_path}\n'
            f'  [y/n]: '
        ).strip().lower()
        cleanup_symlinks = response == 'y'

    if cleanup_symlinks:
        shutil.rmtree(staging_path, ignore_errors=True)
        print(f'  Staging directory removed: {staging_path}')
    else:
        print(f'  Staging directory retained: {staging_path}')

    return trackstats_outfile


# ------------------------------------------------------------------ #
# HELPER: Reconstruct config from an existing staging directory
#         Use this if Step 1 already completed and you need to
#         resume from Step 2 or 3 in a fresh kernel
# ------------------------------------------------------------------ #
def RadarTrackingBuildConfig(
    PeriodStart,
    PeriodEnd,
    RadarIDno,
    QualityControlOption,
    TrackingVersionNumber,
):
    """
    Reconstruct the PyFLEXTRKR config dict from an existing staging
    directory without re-running Step 1 or re-creating symlinks.

    Use this when Step 1 has already completed but the kernel has been
    restarted and the config variable is no longer in memory.
    """
    def parse_period_timestamp(ts_str):
        return pd.Timestamp(ts_str.replace('UTC', ' '), tz=None)

    t_start     = parse_period_timestamp(PeriodStart)
    t_end       = parse_period_timestamp(PeriodEnd)
    start_label = t_start.strftime('%Y%m%d')
    end_label   = t_end.strftime('%Y%m%d')

    QC  = str(QualityControlOption)
    VER = str(TrackingVersionNumber)
    RID = str(RadarIDno)

    staging_path = (
        f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{VER}/QC{QC}/Radar{RID}/'
        f'Symlink_{start_label}_{end_label}/SingleTimes/'
    )
    stats_path = (
        f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/QC{QC}/V{VER}/Radar{RID}/'
        f'Period_{start_label}_{end_label}/Stats/'
    )
    os.makedirs(stats_path, exist_ok=True)

    def datetime_from_cellid(filepath):
        basename = os.path.basename(filepath)
        parts    = basename.replace('.nc', '').split('_')
        return pd.Timestamp(
            f"{parts[1][:4]}-{parts[1][4:6]}-{parts[1][6:8]} "
            f"{parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}"
        )

    all_cellids = sorted(glob.glob(os.path.join(staging_path, 'cellidfile_*.nc')))
    if len(all_cellids) == 0:
        raise RuntimeError(
            f'No cellidfiles found in {staging_path}. '
            f'Has Step 1 been run, and is the staging directory still intact?'
        )

    epoch          = pd.Timestamp('1970-01-01')
    t_first        = datetime_from_cellid(all_cellids[0])
    t_last         = datetime_from_cellid(all_cellids[-1])
    start_basetime = (t_first - epoch).total_seconds()
    end_basetime   = (t_last  - epoch).total_seconds()
    startdate_str  = t_first.strftime('%Y%m%d.%H%M%S')
    enddate_str    = t_last.strftime('%Y%m%d.%H%M%S')

    total_minutes      = (t_last - t_first).total_seconds() / 60.0
    max_duration_steps = int(total_minutes / 5) + 1

    print(f'  Config reconstructed from {len(all_cellids)} cellidfiles')
    print(f'  Period : {startdate_str} → {enddate_str}')
    print(f'  Tracks : {len(sorted(glob.glob(os.path.join(staging_path, "track_*.nc"))))} track files found')

    return {
        'feature_type'                  : 'radar_cells',
        'tracking_outpath'              : staging_path,
        'stats_outpath'                 : stats_path,
        'cloudid_filebase'              : 'cellidfile_',
        'singletrack_filebase'          : 'track_',
        'tracknumbers_filebase'         : 'tracknumbers_',
        'trackstats_filebase'           : 'trackstats_',
        'trackstats_sparse_filebase'    : 'trackstats_sparse_',
        'pixeltracking_filebase'        : 'celltracks_',
        'startdate'                     : startdate_str,
        'enddate'                       : enddate_str,
        'start_basetime'                : start_basetime,
        'end_basetime'                  : end_basetime,
        'time_format'                   : 'yyyymodd.hhmmss',
        'datatimeresolution'            : 5.0 / 60.0,
        'dx'                            : 1000.0,
        'dy'                            : 1000.0,
        'pixel_radius'                  : 1.0,
        'area_method'                   : 'fixed',
        'feature_varname'               : 'feature_number',
        'nfeature_varname'              : 'nfeatures',
        'featuresize_varname'           : 'npix_feature',
        'ref_varname'                   : 'dbz_comp',
        'x_dimname'                     : 'lon',
        'y_dimname'                     : 'lat',
        'time_dimname'                  : 'time',
        'timegap'                       : 0.25,
        'othresh'                       : 0.3,
        'nmaxlinks'                     : 10,
        'maxnclouds'                    : 220,
        'fillval'                       : -9999,
        'duration_range'                : [4, max_duration_steps],
        'duration_range_auto_update'    : True,
        'duration_range_round_base'     : 1,
        'remove_shorttracks'            : 1,
        'trackstats_dense_netcdf'       : 0,
        'match_pixel_dt_thresh'         : 60.0,
        'tracks_dimname'                : 'tracks',
        'times_dimname'                 : 'times',
        'run_parallel'                  : 0,
        'terrain_file'                  : None,
        'rangemask_varname'             : None,
    }


In [5]:
# FUNCTION (CHAD)
# STEPS TWO AND THREE: STAGING DIRECTORY POPULATION

# Symlinks all cellidfile_*.nc and track_*.nc files from per-day
# SingleTimes directories into a single staging directory, ready
# for RadarTrackingStep2 and RadarTrackingStep3.

def RadarBuildStagingDirectory(
    PeriodStart,
    PeriodEnd,
    RadarIDno,
    QualityControlOption,
    TrackingVersionNumber,
):
    """
    Symlink all cellidfile_*.nc and track_*.nc files from per-day
    SingleTimes directories into a single flat staging directory,
    then return the populated config dict ready for Steps 2 and 3.

    The staging directory is the same one that RadarTrackingBuildConfig
    and RadarTrackingStep1 expect, so Steps 2 and 3 can be called
    directly on the returned config.

    Cross-year periods are handled correctly.

    Parameters
    ----------
    PeriodStart           : str  e.g. '2024-01-01UTC00:00:00'
    PeriodEnd             : str  e.g. '2024-02-29UTC23:59:59'
    RadarIDno             : str
    QualityControlOption  : int
    TrackingVersionNumber : int

    Returns
    -------
    config : dict
        PyFLEXTRKR config dict with correct start/end times derived
        from the staged cellidfiles, ready for Steps 2 and 3.
    """

    def parse_period_timestamp(ts_str):
        return pd.Timestamp(ts_str.replace('UTC', ' '), tz=None)

    def datetime_from_cellid(filepath):
        basename = os.path.basename(filepath)
        parts    = basename.replace('.nc', '').split('_')
        return pd.Timestamp(
            f"{parts[1][:4]}-{parts[1][4:6]}-{parts[1][6:8]} "
            f"{parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}"
        )

    t_start = parse_period_timestamp(PeriodStart)
    t_end   = parse_period_timestamp(PeriodEnd)

    if t_end <= t_start:
        raise ValueError(
            f'PeriodEnd ({PeriodEnd}) must be after PeriodStart ({PeriodStart}).'
        )

    QC  = str(QualityControlOption)
    VER = str(TrackingVersionNumber)
    RID = str(RadarIDno)

    start_label = t_start.strftime('%Y%m%d')
    end_label   = t_end.strftime('%Y%m%d')

    base_path = (
        f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{VER}/QC{QC}/Radar{RID}/'
    )

    staging_path = (
        f'{base_path}'
        f'Symlink_{start_label}_{end_label}/SingleTimes/'
    )
    stats_path = (
        f'{base_path}'
        f'Period_{start_label}_{end_label}/Stats/'
    )

    os.makedirs(staging_path, exist_ok=True)
    os.makedirs(stats_path,   exist_ok=True)

    # ------------------------------------------------------------------ #
    # 1. Loop over every day in the period and symlink files
    # ------------------------------------------------------------------ #
    date_range = pd.date_range(
        start = t_start.normalize(),
        end   = t_end.normalize(),
        freq  = 'D',
    )

    all_cellid_symlinks = []
    missing_days        = []
    skipped_cellids     = []
    skipped_tracks      = []

    for day in date_range:
        date_str = day.strftime('%Y%m%d')
        year_int = day.year

        daily_dir = (
            f'{base_path}{year_int}/{date_str}/SingleTimes/'
        )

        # --- cellidfiles ---
        cellid_files = sorted(
            glob.glob(os.path.join(daily_dir, 'cellidfile_*.nc'))
        )

        if len(cellid_files) == 0:
            missing_days.append(date_str)
            print(f'  ⚠ No cellidfiles found for {date_str} in {daily_dir}')
            continue

        for src in cellid_files:
            # Filter to period bounds
            try:
                file_time = datetime_from_cellid(src)
            except (IndexError, ValueError):
                print(f'  ⚠ Could not parse timestamp from {os.path.basename(src)}, skipping.')
                continue

            if not (t_start <= file_time <= t_end):
                continue

            dst = os.path.join(staging_path, os.path.basename(src))
            if os.path.exists(dst) or os.path.islink(dst):
                skipped_cellids.append(dst)
                continue

            os.symlink(src, dst)
            all_cellid_symlinks.append(dst)

        # --- track files ---
        track_files = sorted(
            glob.glob(os.path.join(daily_dir, 'track_*.nc'))
        )

        for src in track_files:
            # Filter to period bounds using same timestamp parser
            # (track files follow the same naming convention as cellidfiles)
            try:
                file_time = datetime_from_cellid(src)
            except (IndexError, ValueError):
                print(f'  ⚠ Could not parse timestamp from {os.path.basename(src)}, skipping.')
                continue

            if not (t_start <= file_time <= t_end):
                continue

            dst = os.path.join(staging_path, os.path.basename(src))
            if os.path.exists(dst) or os.path.islink(dst):
                skipped_tracks.append(dst)
                continue

            os.symlink(src, dst)

    # ------------------------------------------------------------------ #
    # 2. Verify we have something to work with
    # ------------------------------------------------------------------ #
    all_cellid_symlinks = sorted(
        glob.glob(os.path.join(staging_path, 'cellidfile_*.nc'))
    )
    all_track_symlinks = sorted(
        glob.glob(os.path.join(staging_path, 'track_*.nc'))
    )

    if len(all_cellid_symlinks) == 0:
        raise RuntimeError(
            f'No cellidfiles were staged for {PeriodStart} → {PeriodEnd}. '
            f'Check that feature extraction has been run for this period.'
        )

    # ------------------------------------------------------------------ #
    # 3. Print summary
    # ------------------------------------------------------------------ #
    print(f'\n  Period   : {PeriodStart} → {PeriodEnd}')
    print(f'  Staged   : {len(all_cellid_symlinks)} cellidfiles')
    print(f'  Staged   : {len(all_track_symlinks)} track files')
    if skipped_cellids:
        print(f'  Skipped  : {len(skipped_cellids)} cellidfiles (already existed)')
    if skipped_tracks:
        print(f'  Skipped  : {len(skipped_tracks)} track files (already existed)')
    if missing_days:
        print(f'  Missing  : {len(missing_days)} day(s) had no cellidfiles — {missing_days}')
    print()

    # ------------------------------------------------------------------ #
    # 4. Derive config start/end from the actual staged cellidfiles
    # ------------------------------------------------------------------ #
    epoch          = pd.Timestamp('1970-01-01')
    t_first        = datetime_from_cellid(all_cellid_symlinks[0])
    t_last         = datetime_from_cellid(all_cellid_symlinks[-1])
    start_basetime = (t_first - epoch).total_seconds()
    end_basetime   = (t_last  - epoch).total_seconds()
    startdate_str  = t_first.strftime('%Y%m%d.%H%M%S')
    enddate_str    = t_last.strftime('%Y%m%d.%H%M%S')

    total_minutes      = (t_last - t_first).total_seconds() / 60.0
    max_duration_steps = int(total_minutes / 5) + 1

    # ------------------------------------------------------------------ #
    # 5. Build and return config
    # ------------------------------------------------------------------ #
    config = {
        'feature_type'                  : 'radar_cells',
        'tracking_outpath'              : staging_path,
        'stats_outpath'                 : stats_path,
        'cloudid_filebase'              : 'cellidfile_',
        'singletrack_filebase'          : 'track_',
        'tracknumbers_filebase'         : 'tracknumbers_',
        'trackstats_filebase'           : 'trackstats_',
        'trackstats_sparse_filebase'    : 'trackstats_sparse_',
        'pixeltracking_filebase'        : 'celltracks_',
        'startdate'                     : startdate_str,
        'enddate'                       : enddate_str,
        'start_basetime'                : start_basetime,
        'end_basetime'                  : end_basetime,
        'time_format'                   : 'yyyymodd.hhmmss',
        'datatimeresolution'            : 5.0 / 60.0,
        'dx'                            : 1000.0,
        'dy'                            : 1000.0,
        'pixel_radius'                  : 1.0,
        'area_method'                   : 'fixed',
        'feature_varname'               : 'feature_number',
        'nfeature_varname'              : 'nfeatures',
        'featuresize_varname'           : 'npix_feature',
        'ref_varname'                   : 'dbz_comp',
        'x_dimname'                     : 'lon',
        'y_dimname'                     : 'lat',
        'time_dimname'                  : 'time',
        'timegap'                       : 0.25,
        'othresh'                       : 0.3,
        'nmaxlinks'                     : 10,
        'maxnclouds'                    : 1000,
        'fillval'                       : -9999,
        'duration_range'                : [4, max_duration_steps],
        'duration_range_auto_update'    : True,
        'duration_range_round_base'     : 1,
        'remove_shorttracks'            : 1,
        'trackstats_dense_netcdf'       : 1,
        'match_pixel_dt_thresh'         : 60.0,
        'tracks_dimname'                : 'tracks',
        'times_dimname'                 : 'times',
        'run_parallel'                  : 0,
        'terrain_file'                  : None,
        'rangemask_varname'             : None,
    }

    return config


In [ ]:
# EXECUTION (CHAD)
# STEPS ONE, TWO, AND THREE  
# DASK PARALLELIZED W/ PROGRESS BAR
# 3 SEPARATED TRACKING FUNCTIONS

PeriodStart           = '2024-01-01UTC00:00:00'
PeriodEnd             = '2024-01-31UTC23:59:59'
RadarIDno             = '19'
QualityControlOption  = 2
TrackingVersionNumber = 5

# RUN STEP ONE
# config = RadarTrackingStep1( PeriodStart, PeriodEnd, RadarIDno, QualityControlOption, TrackingVersionNumber )

# LOAD IN RESULTS FROM STEP ONE IF YOU DID IT BEFORE
# --- Populate staging directory and get config ---
config = RadarBuildStagingDirectory(
    PeriodStart,
    PeriodEnd,
    RadarIDno,
    QualityControlOption,
    TrackingVersionNumber,
)

# RUN STEP TWO

RadarTrackingStep2(config)

# config = RadarTrackingBuildConfig(
#     PeriodStart           = '2024-02-01UTC00:00:00',
#     PeriodEnd             = '2024-02-29UTC23:59:59',
#     RadarIDno             = '22',
#     QualityControlOption  = 2,
#     TrackingVersionNumber = 5,
# )

# RUN STEP THREE
trackstats_file = RadarTrackingStep3(config, cleanup_symlinks=None)


INFO:pyflextrkr.gettracks:Tracking features sequentially from single track files
INFO:pyflextrkr.ft_utilities:Scanning track files to find maximum number of clouds...
INFO:pyflextrkr.ft_utilities:Scanning 8913 track files...



  Period   : 2024-01-01UTC00:00:00 → 2024-01-31UTC23:59:59
  Staged   : 8917 cellidfiles
  Staged   : 8913 track files

  Step 2/3: Assembling full tracks ...


INFO:pyflextrkr.ft_utilities:Processed 500/8913 files...
INFO:pyflextrkr.ft_utilities:Processed 1000/8913 files...
INFO:pyflextrkr.ft_utilities:Processed 1500/8913 files...
INFO:pyflextrkr.ft_utilities:Processed 2000/8913 files...
INFO:pyflextrkr.ft_utilities:Processed 2500/8913 files...
INFO:pyflextrkr.ft_utilities:Processed 3000/8913 files...
INFO:pyflextrkr.ft_utilities:Processed 3500/8913 files...


In [ ]:
# OLDER VERSIONS BELOW
# OLDER VERSIONS BELOW
# OLDER VERSIONS BELOW
# OLDER VERSIONS BELOW
# OLDER VERSIONS BELOW

# original non-parallelised version

# RETRIEVE FEATURES THAT EXIST AT EACH TIME STEP

logging.basicConfig(level=logging.INFO)

# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
RadarIDno = '22' 

QualityControlOption = 2 # [0 (none) or 1,2, o 3 (int)] # which quality control do you want to grid from?

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 1
RadarDay   = 1
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# USER CHOICE FOLLOW-ON SECTION

# RADAR CHOICE FOLLOW-ON
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton (Brisbane)'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marburg'
elif (RadarIDno == '19'):
    RadarSiteName = 'Cairns'
elif (RadarIDno == '8'):
    RadarSiteName = 'Gympie'
elif (RadarIDno == '24'):
    RadarSiteName = 'Bowen'
elif (RadarIDno == '23'):
    RadarSiteName = 'Gladstone'
elif (RadarIDno == '74'):
    RadarSiteName = 'Greenvale'
elif (RadarIDno == '98'):
    RadarSiteName = 'Taroom'
elif (RadarIDno == '108'):
    RadarSiteName = 'Towoomba'
elif (RadarIDno == '78'):
    RadarSiteName = 'Weipa'
elif (RadarIDno == '72'):
    RadarSiteName = 'Emerald'
else:
    RadarSiteName = 'Site ' + RadarIDno

# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# Longitude shift correction for known coordinate errors
if RadarSiteName == 'Townsville':
    LonShift = 146.5505 - (-19.4195)
else:
    LonShift = 0.0

# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):

    houri = MinOfDay // 60
    mini  = MinOfDay % 60

    RadarFileTime      = str(houri).zfill(2) + str(mini).zfill(2) + '00'
    RadarFileTimePrint = RadarFileTime[0:2] + ':' + RadarFileTime[2:4] + ':' + RadarFileTime[4:6]

    print(f'Working on {RadarFileTimePrint}')

    # RadarGridPath = (
    #     '/scratch/v46/sg3241/tmp/CompressedRadarGrids/'
    #     + RadarIDno + '/' + YYYY + MM + DD + '/'
    #     + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc'
    # )

    RadarGridPath = (
        '/scratch/v46/sg3241/tmp/CustomRadarGrids/'
        + RadarIDno + '/' + YYYY + MM + DD + '/QC' + str(QualityControlOption) + '/'
        + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + 'QC' + str(QualityControlOption) + '_CompLevel4.nc'
    )

    # --- Check file exists without loading it into memory ---
    if not os.path.exists(RadarGridPath):
        print(f'File missing for {RadarFileTimePrint}, skipping: {RadarGridPath}')
        continue

    # --- Extract scheduled date/time from filename ---
    input_basename = os.path.basename(RadarGridPath)
    parts          = input_basename.replace('.nc', '').split('_')
    scheduled_date = parts[1]
    scheduled_time = parts[2]

    # --- Output path ---
    output_path = ('/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/SingleTimes/' + RadarIDno + '/V5/QC' + \
                   str(QualityControlOption) + '/' + YYYY + MM + DD + '/')
    os.makedirs(output_path, exist_ok=True)

    # --- Skip if output already exists ---
    scheduled_outfile = output_path + f'cellidfile_{scheduled_date}_{scheduled_time}.nc'
    if os.path.exists(scheduled_outfile):
        print(f'Already exists, skipping')
        continue

    # --- Config ---
    pyflextrkr_config = {
        'input_format'          : 'netcdf',
        'reflectivity_varname'  : 'corrected_reflectivity',
        'time_dimname'          : 'time',
        'z_dimname'             : 'z',
        'y_dimname'             : 'y',
        'x_dimname'             : 'x',
        'x_coordname'           : 'x',
        'y_coordname'           : 'y',
        'z_coordname'           : 'z',
        'lon_coordname'         : 'lon',
        'lat_coordname'         : 'lat',
        'radar_lon_varname'     : 'radar_longitude',
        'radar_lat_varname'     : 'radar_latitude',
        'dx'                    : 1000.0,
        'dy'                    : 1000.0,
        'is_3d'                 : True,
        'z_coord_type'          : 'height',
        'sfc_dz_min'            : 500.0,
        'sfc_dz_max'            : 3000.0,
        'echotop_gap'           : 2,
        'default_sfc_height'    : 0.0,
        'radar_sensitivity'     : 0.0,
        'absConvThres'          : 30.0,
        'minZdiff'              : 10.0,
        'truncZconvThres'       : 42.0,
        'mindBZuse'             : 10.0,
        'dBZforMaxConvRadius'   : 45.0,
        'conv_rad_increment'    : 0.2,
        'conv_rad_start'        : 1.0,
        'bkg_refl_increment'    : 5.0,
        'maxConvRadius'         : 1.0,
        'radii_expand'          : [1, 2, 3, 4, 5],
        'weakEchoThres'         : 15.0,
        'bkgrndRadius'          : 11.0,
        'min_corearea'          : 4.0,
        'min_cellarea'          : 4.0,
        'tracking_outpath'      : output_path,
        'cloudid_filebase'      : 'cellidfile_tmp_',
        'fillval'               : -9999,
        'return_diag'           : False,
        'convolve_method'       : 'fft',
        'dilate_method'         : 'orig',
        'expand_method'         : 'orig',
        'echotop_method'        : 'orig',
        'remove_smallcores'     : True,
        'remove_smallcells'     : False,
    }

    # --- Run identification ---
    try:
        outfile = idcells_reflectivity(RadarGridPath, pyflextrkr_config)
        shutil.move(outfile, scheduled_outfile)
        print(f'Output written to: {scheduled_outfile}')
    except Exception as e:
        print(f'FAILED for {RadarFileTimePrint}: {e}')

    # --- Explicitly release memory after every timestep ---
    finally:
        gc.collect()

In [ ]:
# FUNNELING IN THE FEATURES FROM INDIVIDUAL TIME STEPS TO GET STATS
sys.path.insert(0, '/scratch/v46/sg3241/tmp/PyFLEXTRKR/')
logging.basicConfig(level=logging.WARNING)


# --- Paths ---
radar_id      = '22'
date_str      = '20240214'
cellid_path   = f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/SingleTimes/QC{QualityControlOption}/Radar{radar_id}/{date_str}/'
tracking_path = f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/Tracking/QC{QualityControlOption}/Radar{radar_id}/{date_str}/'
stats_path    = f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V5/Stats/QC{QualityControlOption}/Radar{radar_id}/{date_str}/'

os.makedirs(tracking_path, exist_ok=True)
os.makedirs(stats_path,    exist_ok=True)

# --- Get start/end times from available cellid files ---
cellid_files = sorted(glob.glob(os.path.join(cellid_path, 'cellidfile_*.nc')))
print(f"Found {len(cellid_files)} cellid files")

def datetime_from_cellid(filepath):
    basename = os.path.basename(filepath)           # cellidfile_20241205_120000.nc
    parts    = basename.replace('.nc', '').split('_')
    return pd.Timestamp(f"{parts[1]} {parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}")

t_start = datetime_from_cellid(cellid_files[0])
t_end   = datetime_from_cellid(cellid_files[-1])

# Convert to epoch seconds (required by tracksingle_driver)
epoch           = pd.Timestamp('1970-01-01')
start_basetime  = (t_start - epoch).total_seconds()
end_basetime    = (t_end   - epoch).total_seconds()

startdate_str   = t_start.strftime('%Y%m%d.%H%M%S')
enddate_str     = t_end.strftime('%Y%m%d.%H%M%S')

print(f"Time range: {t_start} to {t_end}")
print(f"Epoch seconds: {start_basetime} to {end_basetime}")

# --- Full config ---
config = {
    # --- Feature type ---
    'feature_type'                  : 'radar_cells',

    # --- Paths ---
    'tracking_outpath'              : cellid_path,    # where cellid files live
    'stats_outpath'                 : stats_path,

    # --- File bases ---
    'cloudid_filebase'              : 'cellidfile_',
    'singletrack_filebase'          : 'track_',
    'tracknumbers_filebase'         : 'tracknumbers_',
    'trackstats_filebase'           : 'trackstats_',
    'trackstats_sparse_filebase'    : 'trackstats_sparse_',
    'pixeltracking_filebase'        : 'celltracks_',

    # --- Time range ---
    'startdate'                     : startdate_str,
    'enddate'                       : enddate_str,
    'start_basetime'                : start_basetime,
    'end_basetime'                  : end_basetime,
    'time_format'                   : 'yyyymodd.hhmmss',
    'datatimeresolution'            : 5.0 / 60.0,     # hours (5-min data)

    # --- Grid parameters ---
    'dx'                            : 1000.0,          # metres
    'dy'                            : 1000.0,          # metres
    'pixel_radius'                  : 1.0,             # km
    'area_method'                   : 'fixed',

    # --- Variable names in cellid files ---
    'feature_varname'               : 'feature_number',
    'nfeature_varname'              : 'nfeatures',
    'featuresize_varname'           : 'npix_feature',
    'ref_varname'                   : 'dbz_comp',

    # --- Dimension names in cellid files ---
    'x_dimname'                     : 'lon',
    'y_dimname'                     : 'lat',
    'time_dimname'                  : 'time',

    # --- Tracking parameters ---
    'timegap'                       : 0.25,            # hours
    'othresh'                       : 0.3,             # 30% overlap
    'nmaxlinks'                     : 10,
    'maxnclouds'                    : 1000,
    'fillval'                       : -9999,

    # --- Track duration ---
    'duration_range'             : [4, 288],   # min 2 timesteps, max 288 (full day)
    'duration_range_auto_update' : True,      # automatically adjust based on data
    'duration_range_round_base'  : 1,          # round auto-updated duration to nearest 10
    'remove_shorttracks'         : 1,         # remove tracks shorter than duration_range[0]


    # --- Output options ---
    'trackstats_dense_netcdf'       : 1,
    'match_pixel_dt_thresh'         : 60.0,            # seconds

    # --- Dimension names in output files ---
    'tracks_dimname'                : 'tracks',
    'times_dimname'                 : 'times',

    # --- Parallelisation (0=serial, 1=local dask cluster) ---
    'run_parallel'                  : 0,

    # --- No terrain file ---
    'terrain_file'                  : None,
    'rangemask_varname'             : None,
}

# ── Step 1: Pairwise linking ───────────────────────────────────────────────────
print("\n--- Step 1: tracksingle (pairwise linking) ---")
tracksingle_driver(config)
print("tracksingle complete.")

# ── Step 2: Assemble pairs into full tracks ────────────────────────────────────
print("\n--- Step 2: gettracknumbers (assemble full tracks) ---")
gettracknumbers(config)
print("gettracknumbers complete.")

# ── Step 3: Track statistics ───────────────────────────────────────────────────
print("\n--- Step 3: trackstats (compute statistics) ---")
trackstats_outfile = trackstats_driver(config)
print(f"trackstats complete.")
print(f"Output: {trackstats_outfile}")


In [ ]:
# CHAD FUNCTION
# PARALLELISED VERSION OF TRACKING ACROSS TIME STEPS
# (3 STEPS IN ONE)

"""Run PyFLEXTRKR tracking pipeline for a single day."""
def RadarTracking(
    RadarDay, RadarIDno, RadarYear, RadarMonth, QualityControlOption
):
    warnings.filterwarnings('ignore')
    logging.getLogger('pyflextrkr').setLevel(logging.ERROR)

    YYYY = str(RadarYear).zfill(4)
    MM   = str(RadarMonth).zfill(2)
    DD   = str(RadarDay).zfill(2)
    date_str = YYYY + MM + DD

    try:
        # --- Paths ---
        cellid_path   = f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/SingleTimes/{RadarIDno}/{date_str}/QC{QualityControlOption}/V5/'
        stats_path    = f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/Stats/{RadarIDno}/{date_str}/QC{QualityControlOption}/V5/'
        
        os.makedirs(stats_path, exist_ok=True)

        # --- Get start/end times from available cellid files ---
        cellid_files = sorted(glob.glob(os.path.join(cellid_path, 'cellidfile_*.nc')))

        if len(cellid_files) == 0:
            return f'⚠ No cellid files found for {date_str}, skipping'

        def datetime_from_cellid(filepath):
            basename = os.path.basename(filepath)
            parts    = basename.replace('.nc', '').split('_')
            return pd.Timestamp(f"{parts[1]} {parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}")

        t_start = datetime_from_cellid(cellid_files[0])
        t_end   = datetime_from_cellid(cellid_files[-1])

        epoch          = pd.Timestamp('1970-01-01')
        start_basetime = (t_start - epoch).total_seconds()
        end_basetime   = (t_end   - epoch).total_seconds()
        startdate_str  = t_start.strftime('%Y%m%d.%H%M%S')
        enddate_str    = t_end.strftime('%Y%m%d.%H%M%S')

        # --- Config ---
        config = {
            'feature_type'                  : 'radar_cells',
            'tracking_outpath'              : cellid_path,
            'stats_outpath'                 : stats_path,
            'cloudid_filebase'              : 'cellidfile_',
            'singletrack_filebase'          : 'track_',
            'tracknumbers_filebase'         : 'tracknumbers_',
            'trackstats_filebase'           : 'trackstats_',
            'trackstats_sparse_filebase'    : 'trackstats_sparse_',
            'pixeltracking_filebase'        : 'celltracks_',
            'startdate'                     : startdate_str,
            'enddate'                       : enddate_str,
            'start_basetime'                : start_basetime,
            'end_basetime'                  : end_basetime,
            'time_format'                   : 'yyyymodd.hhmmss',
            'datatimeresolution'            : 5.0 / 60.0,
            'dx'                            : 1000.0,
            'dy'                            : 1000.0,
            'pixel_radius'                  : 1.0,
            'area_method'                   : 'fixed',
            'feature_varname'               : 'feature_number',
            'nfeature_varname'              : 'nfeatures',
            'featuresize_varname'           : 'npix_feature',
            'ref_varname'                   : 'dbz_comp',
            'x_dimname'                     : 'lon',
            'y_dimname'                     : 'lat',
            'time_dimname'                  : 'time',
            'timegap'                       : 0.25,
            'othresh'                       : 0.3,
            'nmaxlinks'                     : 10,
            'maxnclouds'                    : 1000,
            'fillval'                       : -9999,
            'duration_range'                : [4, 288],
            'duration_range_auto_update'    : True,
            'duration_range_round_base'     : 1,
            'remove_shorttracks'            : 1,
            'trackstats_dense_netcdf'       : 1,
            'match_pixel_dt_thresh'         : 60.0,
            'tracks_dimname'                : 'tracks',
            'times_dimname'                 : 'times',
            'run_parallel'                  : 0,
            'terrain_file'                  : None,
            'rangemask_varname'             : None,
        }

        # --- Step 1: Pairwise linking ---
        tracksingle_driver(config)

        # --- Step 2: Assemble full tracks ---
        gettracknumbers(config)

        # --- Step 3: Track statistics ---
        trackstats_outfile = trackstats_driver(config)

        return f'✓ Completed {date_str}: {trackstats_outfile}'

    except Exception as e:
        return f'✗ Failed    {date_str}: {str(e)}'

    finally:
        gc.collect()


In [ ]:
# CHAD 
# EXECUTION
# PARALLELISED VERSION OF TRACKING ACROSS TIME STEPS
# (3 STEPS IN ONE)

logging.getLogger().setLevel(logging.WARNING)

# CONFIGURATION
RadarIDno            = '22'
RadarYear            = 2024
RadarMonth           = 2
QualityControlOption = 2

# Build task list — one task per day
tasks_args = [
    (RadarDay, RadarIDno, RadarYear, RadarMonth, QualityControlOption)
    for RadarDay in range(2, 28 + 1)
]

print(f'Total tasks queued: {len(tasks_args)}')

# Each task is one full day — memory per worker is much lower than gridding
# since we are just reading/writing small cellid files, not large radar grids
cluster = LocalCluster(
    n_workers          = 25,
    threads_per_worker = 1,
    memory_limit       = '7GB',
)
client = Client(cluster)

futures = [client.submit(RadarTracking, *args) for args in tasks_args]

for future in tqdm(as_completed(futures), total=len(futures), desc='Tracking Progress'):
    result = future.result()
    if '✗' in result:
        print(result)
    else:
        print(result)   # print all results here since there are only 28 tasks

client.close()


In [ ]:
# CHAD FUNCTION
# FUNNELING TIME STEPS TO TRACK FEATURES OVER TIME
# EXTENDED PERIOD VERSION

def RadarTrackingPeriod(
    PeriodStart,
    PeriodEnd,
    RadarIDno,
    QualityControlOption,
    TrackingVersionNumber,
    cleanup_symlinks = None,   # True / False / None (will prompt if None)
):
    """
    Run PyFLEXTRKR tracking pipeline across an arbitrary UTC period.

    Parameters
    ----------
    PeriodStart          : str  e.g. '2024-02-01UTC00:00:00'
    PeriodEnd            : str  e.g. '2024-02-28UTC23:59:59'
    RadarIDno            : str  e.g. '22'
    QualityControlOption : int  e.g. 2
    TrackingVersionNumber: int  e.g. 5
    cleanup_symlinks     : bool or None
        True  — delete staging symlink directory after run
        False — leave it in place
        None  — prompt the user interactively
    """
    warnings.filterwarnings('ignore')
    logging.getLogger('pyflextrkr').setLevel(logging.INFO)


    # ------------------------------------------------------------------ #
    # 1. Parse start / end times
    # ------------------------------------------------------------------ #
    def parse_period_timestamp(ts_str):
        """Parse a string like '2024-02-01UTC00:00:00' into a pd.Timestamp."""
        return pd.Timestamp(ts_str.replace('UTC', ' '), tz=None)

    t_start = parse_period_timestamp(PeriodStart)
    t_end   = parse_period_timestamp(PeriodEnd)

    if t_end <= t_start:
        raise ValueError(
            f'PeriodEnd ({PeriodEnd}) must be after PeriodStart ({PeriodStart}).'
        )

    # Friendly labels used in directory names
    start_label = t_start.strftime('%Y%m%d')
    end_label   = t_end.strftime('%Y%m%d')

    QC  = str(QualityControlOption)
    VER = str(TrackingVersionNumber)
    RID = str(RadarIDno)

    # ------------------------------------------------------------------ #
    # 2. Build paths
    # ------------------------------------------------------------------ #
    base_single = f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/SingleTimes/{RID}'
    base_stats  = f'/scratch/v46/sg3241/tmp/PyFLEXTRKnetCDFs/Stats/{RID}'

    staging_path = (
        f'{base_single}/V{VER}/QC{QC}/'
        f'Symlink_{start_label}_{end_label}/'
    )
    stats_path = (
        f'{base_stats}/QC{QC}/V{VER}/'
        f'Period_{start_label}_{end_label}/'
    )

    os.makedirs(staging_path, exist_ok=True)
    os.makedirs(stats_path,   exist_ok=True)

    # ------------------------------------------------------------------ #
    # 3. Discover and symlink cellidfiles across every day in the period
    # ------------------------------------------------------------------ #
    date_range = pd.date_range(
        start = t_start.normalize(),   # floor to midnight of start day
        end   = t_end.normalize(),     # floor to midnight of end day
        freq  = 'D',
    )

    def datetime_from_cellid(filepath):
        """Parse cellidfile_YYYYMMDD_HHMMSS.nc → pd.Timestamp."""
        basename = os.path.basename(filepath)
        parts    = basename.replace('.nc', '').split('_')
        # parts[0] = 'cellidfile', parts[1] = YYYYMMDD, parts[2] = HHMMSS
        return pd.Timestamp(
            f"{parts[1][:4]}-{parts[1][4:6]}-{parts[1][6:8]} "
            f"{parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}"
        )

    all_symlinked = []
    missing_days  = []

    for day in date_range:
        date_str  = day.strftime('%Y%m%d')
        daily_dir = (
            f'{base_single}/{date_str}/QC{QC}/V{VER}/'
        )

        daily_files = sorted(
            glob.glob(os.path.join(daily_dir, 'cellidfile_*.nc'))
        )

        if len(daily_files) == 0:
            missing_days.append(date_str)
            print(f'  ⚠ File missing for {date_str} — no cellidfiles found in {daily_dir}')
            continue

        for src in daily_files:

            # Filter to only files within [t_start, t_end]
            try:
                file_time = datetime_from_cellid(src)
            except (IndexError, ValueError):
                print(f'  ⚠ Could not parse timestamp from {os.path.basename(src)}, skipping.')
                continue

            if not (t_start <= file_time <= t_end):
                continue

            dst = os.path.join(staging_path, os.path.basename(src))

            if os.path.islink(dst):
                os.remove(dst)          # replace stale symlinks silently
            os.symlink(src, dst)
            all_symlinked.append(dst)

    if len(all_symlinked) == 0:
        raise RuntimeError(
            f'No cellidfiles found for the period {PeriodStart} → {PeriodEnd}. '
            f'Check that feature extraction has been run for this period.'
        )

    all_symlinked = sorted(all_symlinked)

    print(f'\n  Period  : {PeriodStart} → {PeriodEnd}')
    print(f'  Files   : {len(all_symlinked)} cellidfiles staged')
    if missing_days:
        print(f'  Missing : {len(missing_days)} day(s) had no cellidfiles')
    print()

    # ------------------------------------------------------------------ #
    # 4. Determine actual start / end times from staged files
    # ------------------------------------------------------------------ #
    epoch          = pd.Timestamp('1970-01-01')
    t_first        = datetime_from_cellid(all_symlinked[0])
    t_last         = datetime_from_cellid(all_symlinked[-1])
    start_basetime = (t_first - epoch).total_seconds()
    end_basetime   = (t_last  - epoch).total_seconds()
    startdate_str  = t_first.strftime('%Y%m%d.%H%M%S')
    enddate_str    = t_last.strftime('%Y%m%d.%H%M%S')

    # ------------------------------------------------------------------ #
    # 5. Compute a safe duration_range upper bound
    # ------------------------------------------------------------------ #
    total_minutes      = (t_last - t_first).total_seconds() / 60.0
    max_duration_steps = int(total_minutes / 5) + 1   # 5-min resolution

    # ------------------------------------------------------------------ #
    # 6. PyFLEXTRKR config
    # ------------------------------------------------------------------ #
    config = {
        'feature_type'                  : 'radar_cells',
        'tracking_outpath'              : staging_path,
        'stats_outpath'                 : stats_path,
        'cloudid_filebase'              : 'cellidfile_',
        'singletrack_filebase'          : 'track_',
        'tracknumbers_filebase'         : 'tracknumbers_',
        'trackstats_filebase'           : 'trackstats_',
        'trackstats_sparse_filebase'    : 'trackstats_sparse_',
        'pixeltracking_filebase'        : 'celltracks_',
        'startdate'                     : startdate_str,
        'enddate'                       : enddate_str,
        'start_basetime'                : start_basetime,
        'end_basetime'                  : end_basetime,
        'time_format'                   : 'yyyymodd.hhmmss',
        'datatimeresolution'            : 5.0 / 60.0,
        'dx'                            : 1000.0,
        'dy'                            : 1000.0,
        'pixel_radius'                  : 1.0,
        'area_method'                   : 'fixed',
        'feature_varname'               : 'feature_number',
        'nfeature_varname'              : 'nfeatures',
        'featuresize_varname'           : 'npix_feature',
        'ref_varname'                   : 'dbz_comp',
        'x_dimname'                     : 'lon',
        'y_dimname'                     : 'lat',
        'time_dimname'                  : 'time',
        'timegap'                       : 0.25,
        'othresh'                       : 0.3,
        'nmaxlinks'                     : 10,
        'maxnclouds'                    : 1000,
        'fillval'                       : -9999,
        'duration_range'                : [4, max_duration_steps],
        'duration_range_auto_update'    : True,
        'duration_range_round_base'     : 1,
        'remove_shorttracks'            : 1,
        'trackstats_dense_netcdf'       : 1,
        'match_pixel_dt_thresh'         : 60.0,
        'tracks_dimname'                : 'tracks',
        'times_dimname'                 : 'times',
        'run_parallel'                  : 0,
        'terrain_file'                  : None,
        'rangemask_varname'             : None,
    }

    # ------------------------------------------------------------------ #
    # 7. Run the three pipeline steps
    # ------------------------------------------------------------------ #
    try:
        t0 = time.time()
        print('  Step 1/3: Pairwise linking ...')
        tracksingle_driver(config)
        print(f'            Done ({time.time() - t0:.1f}s)')

        t0 = time.time()
        print('  Step 2/3: Assembling full tracks ...')
        gettracknumbers(config)
        print(f'            Done ({time.time() - t0:.1f}s)')

        t0 = time.time()
        print('  Step 3/3: Computing track statistics ...')
        trackstats_outfile = trackstats_driver(config)
        print(f'            Done ({time.time() - t0:.1f}s)')

        print(f'\n  ✓ Tracking complete: {trackstats_outfile}\n')
        success = True

    except Exception as e:
        print(f'\n  ✗ Tracking failed: {str(e)}\n')
        trackstats_outfile = None
        success = False

    finally:
        gc.collect()

    # ------------------------------------------------------------------ #
    # 8. Symlink cleanup
    # ------------------------------------------------------------------ #
    if cleanup_symlinks is None:
        response = input(
            f'  Delete staging symlink directory?\n'
            f'  {staging_path}\n'
            f'  [y/n]: '
        ).strip().lower()
        cleanup_symlinks = response == 'y'

    if cleanup_symlinks:
        shutil.rmtree(staging_path, ignore_errors=True)
        print(f'  Staging directory removed: {staging_path}')
    else:
        print(f'  Staging directory retained: {staging_path}')

    return trackstats_outfile if success else None


In [ ]:
# CHAD EXECUTION BLOCK
# FUNNELING TIME STEPS TO TRACK FEATURES OVER TIME
# EXTENDED PERIOD VERSION

# CONFIGURATION
PeriodStart          = '2024-02-01UTC18:00:00'
PeriodEnd            = '2024-02-29UTC23:59:59'
RadarIDno            = '22'
QualityControlOption = 2
TrackingVersionNumber = 5

trackstats_file = RadarTrackingPeriod(
    PeriodStart           = PeriodStart,
    PeriodEnd             = PeriodEnd,
    RadarIDno             = RadarIDno,
    QualityControlOption  = QualityControlOption,
    TrackingVersionNumber = TrackingVersionNumber,
    cleanup_symlinks      = None,   # None = prompt, True = auto-delete, False = auto-keep
)


In [ ]:
import inspect
from pyflextrkr.tracksingle import tracksingle_driver
print(inspect.getsource(tracksingle_driver))


In [ ]:
# CHAD EXECUTION BLOCK
# FOR 3 SEPARATED TRACKING FUNCTIONS

PeriodStart           = '2024-01-01UTC00:00:00'
PeriodEnd             = '2024-01-31UTC23:59:59'
RadarIDno             = '22'
QualityControlOption  = 2
TrackingVersionNumber = 5

# RUN STEP ONE
config = RadarTrackingStep1( PeriodStart, PeriodEnd, RadarIDno, QualityControlOption, TrackingVersionNumber )

# LOAD IN RESULTS FROM STEP ONE IF IT CRASHED
# config = RadarTrackingBuildConfig(
#     PeriodStart, PeriodEnd, RadarIDno, QualityControlOption, TrackingVersionNumber
# )

# # RUN STEP TWO

# RadarTrackingStep2(config)

# # config = RadarTrackingBuildConfig(
# #     PeriodStart           = '2024-02-01UTC00:00:00',
# #     PeriodEnd             = '2024-02-29UTC23:59:59',
# #     RadarIDno             = '22',
# #     QualityControlOption  = 2,
# #     TrackingVersionNumber = 5,
# # )

# # RUN STEP THREE
# trackstats_file = RadarTrackingStep3(config, cleanup_symlinks=None)


In [ ]:
import glob
import xarray as xr

# Check how many tracks Step 2 found
tracknumbers_file = '/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/Stats/22/QC2/V5/Period_20240201_20240229/tracknumbers_20240201.000000_20240229.235500.nc'

ds = xr.open_dataset(tracknumbers_file)
print(ds)
print(f'Number of tracks : {ds.dims.get("tracks", "unknown")}')
print(f'Number of times  : {ds.dims.get("times", "unknown")}')
ds.close()


In [ ]:
import xarray as xr

tracknumbers_file = '/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/Stats/22/QC2/V5/Period_20240201_20240229/tracknumbers_20240201.000000_20240229.235500.nc'

ds = xr.open_dataset(tracknumbers_file)
n_tracks = int(ds['ntracks'].values[0])
n_times  = int(ds.dims['nfiles'])
print(f'Total unique tracks : {n_tracks:,}')
print(f'Total timesteps     : {n_times:,}')
ds.close()


In [ ]:
import xarray as xr
import numpy as np

ds = xr.open_dataset(tracknumbers_file)
track_numbers = ds['track_numbers'].values
max_simultaneous = np.sum(track_numbers[0] > 0, axis=1).max()
print(f'Maximum simultaneous features in any timestep: {max_simultaneous}')
ds.close()

In [ ]:
# CHAD PLOT
# TO DEMONSTRATE THE RELATIONSHIP BETWEEN
# BACKGROUND REFLECTIVITY AND 'PEAKEDNESS'
# IN THE PYFLEXTRKR ALGORITHM


import numpy as np
import matplotlib.pyplot as plt

# ── Parameters to experiment with ─────────────────────────────────────────────
minZdiff     = 8    # Maximum peakedness requirement (dBZ)
absConvThres = 40   # Background dBZ at which requirement drops to zero

# ── Background reflectivity axis ───────────────────────────────────────────────
refl_bkg = np.linspace(-10, 55, 500)

# ── Peakedness curve (replicating the steiner_func logic exactly) ──────────────
peak = np.zeros_like(refl_bkg)
peak[refl_bkg < 0]                = minZdiff
peak[refl_bkg >= 0]               = minZdiff * np.cos(np.pi * refl_bkg[refl_bkg >= 0] / (2 * absConvThres))
peak[refl_bkg >= absConvThres]    = 0
peak[peak < 0]                    = 0   # cosine can go negative beyond absConvThres; clip to 0

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5), facecolor='black')
ax.set_facecolor('black')

ax.plot(refl_bkg, peak, color='orange', linewidth=2.5)

# Annotate key points
ax.axvline(0,            color='white', linestyle='--', linewidth=0.8, alpha=0.5, label='Background = 0 dBZ')
ax.axvline(absConvThres, color='cyan',  linestyle='--', linewidth=0.8, alpha=0.5, label=f'absConvThres = {absConvThres} dBZ')
ax.axhline(minZdiff,     color='red',   linestyle='--', linewidth=0.8, alpha=0.5, label=f'minZdiff = {minZdiff} dBZ')

# Style
ax.set_xlabel('Background Reflectivity (dBZ)', color='white')
ax.set_ylabel('Required Excess Above Background (dBZ)', color='white')
ax.set_title('Peakedness Requirement vs Background Reflectivity', color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('white')
ax.legend(facecolor='black', labelcolor='white', edgecolor='white')
ax.grid(True, color='white', linestyle='--', linewidth=0.5, alpha=0.4)

ax.set_xlim(refl_bkg[0], refl_bkg[-1])
ax.set_ylim(-1, minZdiff + 5)

plt.tight_layout()
plt.show()


In [ ]:
FeaturesXR = xr.open_dataset('/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/SingleTimes/22/20240309/cellidfile_20240309_120000.nc')

In [ ]:
FeaturesXR

In [ ]:
feature_number = FeaturesXR2['feature_number'].isel(time=0).values
npix_feature   = FeaturesXR2['npix_feature'].values
nfeatures      = int(FeaturesXR2['nfeatures'].values[0])

print(f"Number of features detected: {nfeatures}")
print(f"Pixels per feature: {npix_feature}")
print(f"Areas in km²: {npix_feature}  (1 pixel = 1 km²)")

# Quick plot of the feature label map
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(feature_number, origin='lower', cmap='tab20')
ax.set_title(f'Feature label map — {nfeatures} features detected')
plt.tight_layout()
plt.show()

In [ ]:
# test_path = '/scratch/v46/sg3241/tmp/CompressedRadarGrids/22/2024/12/05/22_20241205_120000.nc'
# ds = xr.open_dataset(test_path)

test_path = '/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/Stats/22/V352/20240309/22_20240309FeatureStats.nc'
FeatureStatsXR = xr.open_dataset(test_path)

In [ ]:
FeatureStatsXR 

In [ ]:
# FEATURE TRACKS PLOTTING

# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
RadarIDno = '22' 

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# USER CHOICE FOLLOW-ON SECTION

# RADAR CHOICE FOLLOW-ON
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton (Brisbane)'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marburg'
elif (RadarIDno == '19'):
    RadarSiteName = 'Cairns'
elif (RadarIDno == '8'):
    RadarSiteName = 'Gympie'
elif (RadarIDno == '24'):
    RadarSiteName = 'Bowen'
elif (RadarIDno == '23'):
    RadarSiteName = 'Gladstone'
elif (RadarIDno == '74'):
    RadarSiteName = 'Greenvale'
elif (RadarIDno == '98'):
    RadarSiteName = 'Taroom'
elif (RadarIDno == '108'):
    RadarSiteName = 'Towoomba'
elif (RadarIDno == '78'):
    RadarSiteName = 'Weipa'
elif (RadarIDno == '72'):
    RadarSiteName = 'Emerald'
else:
    RadarSiteName = 'Site ' + RadarIDno

# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# Longitude shift correction for known coordinate errors
if RadarSiteName == 'Townsville':
    LonShift = 146.5505 - (-19.4195)
else:
    LonShift = 0.0
    # ── User settings ──────────────────────────────────────────────────────────────
CoarsenLevel       = 1       # DEM coarsening factor
min_duration_steps = 0       # minimum track duration (timesteps) to plot
# ──────────────────────────────────────────────────────────────────────────────

RadarFileTime = '000000'

# LOAD IN AN EXAMPLE RADAR FILE TO GRAB LAT AND LON LIMITS FROM
RadarGridsFolder = 'CustomadarGrids'
NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/' + RadarGridsFolder + '/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
                                                         + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
# try to load in the netcdf file and if it doesn't work, just keep going through the loop
try:
    xgrid = xr.open_dataset(NetCDFstoragePath)
except FileNotFoundError:
    print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')

fig, ax = plt.subplots(figsize=(8, 6),
                       subplot_kw={'projection': ccrs.PlateCarree()})

# TERRAIN SHADING USING LOCAL GEBCO DEM
lon_min, lon_max = float(xgrid.lon.min()) + LonShift, float(xgrid.lon.max()) + LonShift
lat_min, lat_max = float(xgrid.lat.min()), float(xgrid.lat.max())

try:
    gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)
    if dem_da is None:
        raise ValueError('GEBCO DEM returned None')
    dem_da   = dem_da.coarsen(lon=CoarsenLevel, lat=CoarsenLevel, boundary='trim').mean()
    dem_lon  = dem_da.lon.values
    dem_lat  = dem_da.lat.values
    dem_data = dem_da.values
    if dem_lon.ndim == 1 and dem_lat.ndim == 1:
        dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
    else:
        dem_lon_2d, dem_lat_2d = dem_lon, dem_lat
    if not np.any(np.isfinite(dem_data)):
        raise ValueError('DEM has no finite values in this domain')

    colours = [
        '#dde4e8',
        '#c4dec2', '#e4edc9', '#f3f0cf',
        '#e9d7bd', '#ddc4aa', '#cfb194', '#b58f6e',
    ]
    bounds = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
    cmap_elev = ListedColormap(colours)
    norm_elev = BoundaryNorm(bounds, len(colours), clip=True)
    ax.pcolormesh(dem_lon_2d, dem_lat_2d, dem_data,
                  cmap=cmap_elev, norm=norm_elev, alpha=0.8,
                  transform=ccrs.PlateCarree())
    ax.contour(dem_lon_2d, dem_lat_2d, dem_data,
               levels=[0.0], colors='black', linewidths=0.5,
               transform=ccrs.PlateCarree(), zorder=15)

        # Draw 0 m contour as an accurate coastline
    coast_contour = ax.contour(
        dem_lon_2d,
        dem_lat_2d,
        dem_data,
        levels=[0.0],
        colors='black',
        linewidths=0.5,
        transform=ccrs.PlateCarree(),
        zorder=15,  # above radar and topo
    )
    
    # Draw 400 m contour
    coast_contour = ax.contour(
        dem_lon_2d,
        dem_lat_2d,
        dem_data,
        levels=[400.0],
        colors='black',
        linewidths=0.3,
        transform=ccrs.PlateCarree(),
        zorder=15,  # above radar and topo
    )

except Exception as e:
    print(f'Terrain shading failed: {e}')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.30, zorder=1)
    ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.30, zorder=2)

# ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)


# plot a star for the location of the radar on the map
ax.plot(float(xgrid.radar_longitude) + LonShift, float(xgrid.radar_latitude),
        marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=20)
ax.plot(float(xgrid.radar_longitude) + LonShift, float(xgrid.radar_latitude),
        marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=21)

# ── GRIDLINES ─────────────────────────────────────────────────────────────────

# Add MINOR gridlines (tenth degrees) - thin
gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

# Add MID LEVEL gridlines (half degrees) - standard width with labels
gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

# Add MAJOR gridlines (full degrees) - thick with labels
gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees

# Format labels
gl_mid.xformatter = LONGITUDE_FORMATTER
gl_mid.yformatter = LATITUDE_FORMATTER

gl_major.xformatter = LONGITUDE_FORMATTER
gl_major.yformatter = LATITUDE_FORMATTER

# Remove labels from top and right
gl_mid.top_labels = False
gl_mid.right_labels = False
gl_mid.bottom_labels = True
gl_mid.left_labels = True

gl_major.top_labels = False
gl_major.right_labels = False
gl_major.bottom_labels = True
gl_major.left_labels = True

# ── TRACK PLOTTING ────────────────────────────────────────────────────────────
# Colourmap: time of day mapped to 0–1 (midnight=0, midnight=1)
cmap_tracks = cm.nipy_spectral
norm_tracks = mcolors.Normalize(vmin=0, vmax=24)   # hours of day

n_tracks_plotted = 0

for i in range(len(FeatureStatsXR['track_duration'])):
    duration = int(FeatureStatsXR['track_duration'].values[i])

    # Skip very short tracks
    if duration < min_duration_steps:
        continue

    # Extract valid timesteps only
    lats = FeatureStatsXR['cell_meanlat'].values[i, :duration]
    lons = FeatureStatsXR['cell_meanlon'].values[i, :duration]
    times = FeatureStatsXR['base_time'].values[i, :duration]

    # Skip if all positions are NaN
    if np.all(np.isnan(lats)) or np.all(np.isnan(lons)):
        continue

    # Convert times to hour-of-day (0–24) for colouring
    times_pd    = pd.DatetimeIndex(times)
    hours_of_day = times_pd.hour + times_pd.minute / 60.0 + times_pd.second / 3600.0

    # Plot each segment individually so colour varies along the track
    for j in range(len(lons) - 1):
        if np.isnan(lons[j]) or np.isnan(lats[j]):
            continue
        if np.isnan(lons[j+1]) or np.isnan(lats[j+1]):
            continue
        colour = cmap_tracks(norm_tracks(hours_of_day[j]))
        ax.plot(
            [lons[j], lons[j+1]],
            [lats[j], lats[j+1]],
            color=colour,
            linewidth=0.8,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=20,
        )

    n_tracks_plotted += 1

# ── COLOURBAR ─────────────────────────────────────────────────────────────────
sm = cm.ScalarMappable(cmap=cmap_tracks, norm=norm_tracks)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical',
                    pad=0.02, shrink=0.7, aspect=25)
cbar.set_label('Time of day (UTC hour)', fontsize=9)
cbar.set_ticks([0, 3, 6, 9, 12, 15, 18, 21, 24])
cbar.set_ticklabels(['00:00', '03:00', '06:00', '09:00',
                     '12:00', '15:00', '18:00', '21:00', '24:00'])

# ── EXTENT AND TITLE ──────────────────────────────────────────────────────────
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.set_title('Feature Tracks for ' + RadarSiteName + ' Radar\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' UTC')
plt.tight_layout()

# ── save ──────────────────────────────────────────────────────────────────────
SaveFolder = ('/scratch/v46/sg3241/tmp/pngImages/FeatureTracks/' +
              RadarIDno + '/' + RadarFileDate + '/')
SaveFile = RadarIDno + '_' + RadarFileDate + '_FeatureTracks_PyFLEXTRKRV352.png'

SavePath   = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi=300)
plt.close()
print(f"Saved to {SavePath}")